# v2 Persona Vector Pipeline — Method Comparison

Pure numpy/sklearn on the cache written by `01_extract_activations.ipynb` — no GPU needed, runs anywhere (including a laptop) as long as `PV2_CACHE_DIR` points at that cache (a local copy, or a mounted Drive folder).

Produces, per model: accuracy + bootstrap CI for all three methods on sub-test A and sub-test B's reportable neutral arm, the A/B gap (the spec's win condition), the TF-IDF floor, a random-direction null check, the length-correlation flag for every reported number, and the probe-vs-cosine comparison for whichever method has the smallest A/B gap. Models whose cache isn't present yet are skipped with a clear message, not silently omitted — this notebook is meant to be re-run incrementally as `01_extract_activations.ipynb` finishes each model.

In [1]:
import os
def show_tree(path, depth=3, _d=0):
    try:
        entries = sorted(os.listdir(path))
    except PermissionError:
        return
    for e in entries:
        print("  " * _d + e)
        if _d < depth:
            full = os.path.join(path, e)
            if os.path.isdir(full):
                show_tree(full, depth, _d + 1)

show_tree("/kaggle/input", depth=3)

datasets
  fjordhauler
    pineline-v2-data
      harmbench_filtered_250.csv
      neutral_set_300.csv
      splits.json
      subtest_b_harmful_tone_pairs.csv
      subtest_b_neutral_tone_pairs.csv
      tone_pole_questions_40.csv
      tone_pole_system_prompts.csv
    pipeline-v2-zip
      Pipeline_v2
notebooks
  fjordhauler
    01-extract-activations-gpt2-medium-qwen2-5-1-5
      Persona-Vector-Study
      __huggingface_repos__.json
      __notebook__.ipynb
      __output__.json
      __results__.html
      custom.css
      pv2_cache
      pv2_manifests
    01-extract-activations-llama-3-2-3b-qwen2-5-1-5b-i
      Persona-Vector-Study
      __huggingface_repos__.json
      __notebook__.ipynb
      __output__.json
      __results__.html
      custom.css
      pv2_cache
      pv2_manifests
    pipeline-v2
      Persona-Vector-Study
      __huggingface_repos__.json
      __notebook__.ipynb
      __output__.json
      __results__.html
      custom.css
      pv2_cache
      pv2_manifests


In [2]:
import os, sys, pathlib

NOTEBOOK_INPUT   = pathlib.Path("/kaggle/input/notebooks/fjordhauler/pipeline-v2")
NOTEBOOK_INPUT_2 = pathlib.Path("/kaggle/input/notebooks/fjordhauler/01-extract-activations-gpt2-medium-qwen2-5-1-5")
NOTEBOOK_INPUT_3 = pathlib.Path("/kaggle/input/notebooks/fjordhauler/01-extract-activations-llama-3-2-3b-qwen2-5-1-5b-i")

REPO_DATA    = NOTEBOOK_INPUT / "Persona-Vector-Study/data"
PIPELINE_DIR = pathlib.Path("/kaggle/input/datasets/fjordhauler/pipeline-v2-zip/Pipeline_v2")

sys.path.insert(0, str(PIPELINE_DIR))
os.chdir(PIPELINE_DIR)

os.environ["PV2_DATA_DIR"]    = str(REPO_DATA)
os.environ["PV2_SPLITS_JSON"] = str(REPO_DATA / "splits.json")

from src import config, splits, activation_store
from src.config import MODEL_REGISTRY

config.DATA_DIR              = REPO_DATA
config.HARMBENCH_CSV         = REPO_DATA / "harmbench_filtered_250.csv"
config.NEUTRAL_CSV           = REPO_DATA / "neutral_set_300.csv"
config.SUBTEST_B_NEUTRAL_CSV = REPO_DATA / "subtest_b_neutral_tone_pairs.csv"
config.SUBTEST_B_HARMFUL_CSV = REPO_DATA / "subtest_b_harmful_tone_pairs.csv"
config.TONE_POLE_QUESTIONS_CSV      = REPO_DATA / "tone_pole_questions_40.csv"
config.TONE_POLE_SYSTEM_PROMPTS_CSV = REPO_DATA / "tone_pole_system_prompts.csv"
config.SPLITS_JSON_PATH = REPO_DATA / "splits.json"

import src.splits as _splits
_splits.HARMBENCH_CSV         = config.HARMBENCH_CSV
_splits.NEUTRAL_CSV           = config.NEUTRAL_CSV
_splits.SPLITS_JSON_PATH      = config.SPLITS_JSON_PATH
_splits.SUBTEST_B_NEUTRAL_CSV = config.SUBTEST_B_NEUTRAL_CSV
_splits.SUBTEST_B_HARMFUL_CSV = config.SUBTEST_B_HARMFUL_CSV

import src.eval.subtest_b as _stb
_stb.SUBTEST_B_NEUTRAL_CSV = config.SUBTEST_B_NEUTRAL_CSV
_stb.SUBTEST_B_HARMFUL_CSV = config.SUBTEST_B_HARMFUL_CSV

CACHE_DIR_1 = NOTEBOOK_INPUT   / "pv2_cache"
CACHE_DIR_2 = NOTEBOOK_INPUT_2 / "pv2_cache"
CACHE_DIR_3 = NOTEBOOK_INPUT_3 / "pv2_cache"

for name, p in [
    ("cache_1",   CACHE_DIR_1),
    ("cache_2",   CACHE_DIR_2),
    ("cache_3",   CACHE_DIR_3),
    ("splits",    config.SPLITS_JSON_PATH),
    ("harmbench", config.HARMBENCH_CSV),
    ("neutral",   config.NEUTRAL_CSV),
    ("stb_neut",  config.SUBTEST_B_NEUTRAL_CSV),
]:
    print(f"{name:12s}: {p.exists()}  {p}")

MODEL_CACHE_MAP = {}
for model_key in MODEL_REGISTRY:
    for cache_dir in [CACHE_DIR_1, CACHE_DIR_2, CACHE_DIR_3]:
        if (cache_dir / model_key).exists():
            MODEL_CACHE_MAP[model_key] = cache_dir
            break

print("\nmodel -> cache mapping:")
for k, v in MODEL_CACHE_MAP.items():
    print(f"  {k}: {v.parent.name}")

cache_1     : True  /kaggle/input/notebooks/fjordhauler/pipeline-v2/pv2_cache
cache_2     : True  /kaggle/input/notebooks/fjordhauler/01-extract-activations-gpt2-medium-qwen2-5-1-5/pv2_cache
cache_3     : True  /kaggle/input/notebooks/fjordhauler/01-extract-activations-llama-3-2-3b-qwen2-5-1-5b-i/pv2_cache
splits      : True  /kaggle/input/notebooks/fjordhauler/pipeline-v2/Persona-Vector-Study/data/splits.json
harmbench   : True  /kaggle/input/notebooks/fjordhauler/pipeline-v2/Persona-Vector-Study/data/harmbench_filtered_250.csv
neutral     : True  /kaggle/input/notebooks/fjordhauler/pipeline-v2/Persona-Vector-Study/data/neutral_set_300.csv
stb_neut    : True  /kaggle/input/notebooks/fjordhauler/pipeline-v2/Persona-Vector-Study/data/subtest_b_neutral_tone_pairs.csv

model -> cache mapping:
  gpt2-medium: 01-extract-activations-gpt2-medium-qwen2-5-1-5
  qwen2.5-1.5b: 01-extract-activations-gpt2-medium-qwen2-5-1-5
  qwen2.5-1.5b-instruct: 01-extract-activations-llama-3-2-3b-qwen2-5-1-5b-

## TF-IDF baseline (spec requirement 4) — model-independent, computed once

In [3]:
import numpy as np
from src import splits
from src.methods import tfidf_baseline

splits_df = splits.load_frozen_splits()
text_by_id = {row.prompt_id: row.text for row in splits.load_combined_rows()}

def _ids_labels(split_name):
    subset = splits_df[splits_df["split"] == split_name]
    return subset["PromptID"].tolist(), subset["label"].tolist()

train_ids, train_labels = _ids_labels("train")
test_ids, test_labels = _ids_labels("test")
all_ids, all_labels = splits_df["PromptID"].tolist(), splits_df["label"].tolist()

tfidf_single = tfidf_baseline.fit_and_score_single(
    [text_by_id[i] for i in train_ids], np.array(train_labels),
    [text_by_id[i] for i in test_ids], np.array(test_labels),
)
tfidf_cv = tfidf_baseline.cross_val_score_5fold([text_by_id[i] for i in all_ids], np.array(all_labels))

print(f"TF-IDF single-fit accuracy: {tfidf_single['accuracy']:.4f} (n_train={tfidf_single['n_train']}, n_test={tfidf_single['n_test']})")
print(f"TF-IDF 5-fold CV accuracy:  {tfidf_cv['mean_accuracy']:.4f} +/- {tfidf_cv['std_accuracy']:.4f}")
print("(v1's reference numbers were 66.39% single-fit / 76.17% +/- 2.07 CV -- not expected to match exactly, different dataset, but should be in the same neighborhood as a sanity check)")

TF-IDF single-fit accuracy: 0.9342 (n_train=392, n_test=76)
TF-IDF 5-fold CV accuracy:  0.9655 +/- 0.0235
(v1's reference numbers were 66.39% single-fit / 76.17% +/- 2.07 CV -- not expected to match exactly, different dataset, but should be in the same neighborhood as a sanity check)


In [4]:
# TF-IDF on sub-test B neutral arm — does surface-form collapse where activations don't?
import pandas as pd
from src.methods import tfidf_baseline
from src import splits as splits_mod

stb_neutral = pd.read_csv(config.SUBTEST_B_NEUTRAL_CSV)

# For each pair, train on the full sub-test A train split, score on both calm and hostile variants
train_ids_l, train_labels_l = _ids_labels("train")
train_texts = [text_by_id[i] for i in train_ids_l]

calm_texts    = stb_neutral["calm_prompt"].tolist()
hostile_texts = stb_neutral["hostile_prompt"].tolist()

# Label: all neutral content → should predict "neutral"
calm_labels    = ["neutral"] * len(calm_texts)
hostile_labels = ["neutral"] * len(hostile_texts)

calm_result    = tfidf_baseline.fit_and_score_single(
    train_texts, np.array(train_labels_l),
    calm_texts,  np.array(calm_labels),
)
hostile_result = tfidf_baseline.fit_and_score_single(
    train_texts, np.array(train_labels_l),
    hostile_texts, np.array(hostile_labels),
)

# Combined (same as what the activation methods see)
combined_texts  = calm_texts + hostile_texts
combined_labels = calm_labels + hostile_labels
combined_result = tfidf_baseline.fit_and_score_single(
    train_texts, np.array(train_labels_l),
    combined_texts, np.array(combined_labels),
)

print(f"TF-IDF on sub-test B neutral — calm variants:    {calm_result['accuracy']:.4f}")
print(f"TF-IDF on sub-test B neutral — hostile variants: {hostile_result['accuracy']:.4f}")
print(f"TF-IDF on sub-test B neutral — combined:         {combined_result['accuracy']:.4f}")
print(f"(activation content_pole on qwen2.5-1.5b-instruct/chat was 0.9583 for comparison)")

TF-IDF on sub-test B neutral — calm variants:    0.8333
TF-IDF on sub-test B neutral — hostile variants: 0.8750
TF-IDF on sub-test B neutral — combined:         0.8542
(activation content_pole on qwen2.5-1.5b-instruct/chat was 0.9583 for comparison)


## Length-balance note (spec deliverable 5)

In [5]:
lengths = np.array([len(text_by_id[i]) for i in all_ids])
is_harmful = np.array([1.0 if lbl == "harmful" else 0.0 for lbl in all_labels])
r_length_label = float(np.corrcoef(lengths, is_harmful)[0, 1])
print(f"r(length, label) across the combined 550-prompt set: {r_length_label:.3f}")
print("(manifests report 0.436 at authoring time -- this recomputes it directly from the frozen split's own text)")
print("Per spec, this makes centering/standardization/last-token pooling a hard requirement, not an optional ablation.")
print("Per-method, per-pooling-variant r(score, length) is reported below in each method's length_correlation block --")
print("any result with flagged=True should be treated as unreliable for reporting, the same way v1's mean-pooled results were.")

r(length, label) across the combined 550-prompt set: 0.436
(manifests report 0.436 at authoring time -- this recomputes it directly from the frozen split's own text)
Per spec, this makes centering/standardization/last-token pooling a hard requirement, not an optional ablation.
Per-method, per-pooling-variant r(score, length) is reported below in each method's length_correlation block --
any result with flagged=True should be treated as unreliable for reporting, the same way v1's mean-pooled results were.


## Per-model method comparison: sub-test A, sub-test B, A/B gap, random-direction control

In [6]:
import pandas as pd
import numpy as np
from src.eval import subtest_a as subtest_a_mod
from src.eval import subtest_b as subtest_b_mod
from src.methods import content_pole as _cp
from src.methods import mean_diff as _md

# Patch fit_content_pole to skip degenerate layers (identical class means)
_orig_fit_mean_diff = _md.fit_mean_diff_pole
def _safe_fit_mean_diff(positive_matrix, negative_matrix):
    try:
        return _orig_fit_mean_diff(positive_matrix, negative_matrix)
    except ValueError:
        return None
_md.fit_mean_diff_pole = _safe_fit_mean_diff

_orig_fit_content_pole = _cp.fit_content_pole
def _safe_fit_content_pole(harmful, neutral):
    try:
        return _orig_fit_content_pole(harmful, neutral)
    except (ValueError, TypeError):
        return None
_cp.fit_content_pole = _safe_fit_content_pole

# Also patch the layer loop to skip None poles
import src.eval.subtest_a as _sta
_orig_run = _sta.run_subtest_a

def _patched_run_subtest_a(cache_dir, model_key, formatting_variant, seed=42, include_tone_pole=True):
    import src.activation_store as activation_store
    import src.splits as splits_mod
    from src.config import POOLING_VARIANTS
    from src.eval.bootstrap import bootstrap_accuracy_ci
    from src.eval.length_correlation import length_correlation_report
    from src.methods import content_pole, neutral_origin, random_direction, tone_pole
    from src.eval.subtest_a import FittedMethodResult, _accuracy, _ids_and_labels

    splits_df = splits_mod.load_frozen_splits(seed=seed)
    text_by_id = {row.prompt_id: row.text for row in splits_mod.load_combined_rows()}

    train_harmful_ids, train_neutral_ids, _ = _ids_and_labels(splits_df, "train")
    val_harmful_ids, val_neutral_ids, val_ids = _ids_and_labels(splits_df, "val")
    test_harmful_ids, test_neutral_ids, test_ids = _ids_and_labels(splits_df, "test")

    val_labels = ["harmful"] * len(val_harmful_ids) + ["neutral"] * len(val_neutral_ids)
    test_labels = ["harmful"] * len(test_harmful_ids) + ["neutral"] * len(test_neutral_ids)
    n_layers = activation_store.available_layers(cache_dir, model_key, formatting_variant)

    candidates = {"content_pole": [], "neutral_origin_distance": [], "neutral_origin_direction": []}
    if include_tone_pole:
        candidates["tone_pole"] = []

    for pooling_variant in POOLING_VARIANTS:
        for layer in range(n_layers):
            load = lambda ids: activation_store.load_layer_matrix(
                cache_dir, model_key, formatting_variant, pooling_variant, layer, ids)
            train_harmful = load(train_harmful_ids)
            train_neutral = load(train_neutral_ids)
            val_harmful   = load(val_harmful_ids)
            val_matrix    = load(val_ids)

            # Method 2 -- content pole (skip degenerate layers)
            try:
                pole2 = content_pole.fit_content_pole(train_harmful, train_neutral)
                if pole2 is not None:
                    val_pred2 = content_pole.predict(pole2, val_matrix)
                    candidates["content_pole"].append(FittedMethodResult(
                        method_name="content_pole", pooling_variant=pooling_variant, layer=layer,
                        predict_fn=lambda a, p=pole2: content_pole.predict(p, a),
                        score_fn=lambda a, p=pole2: content_pole.score(p, a),
                        val_accuracy=_accuracy(val_pred2, val_labels),
                        reference_point=pole2.midpoint,
                    ))
            except Exception as e:
                print(f"  content_pole layer {layer} skipped: {e}")

            # Method 3a -- neutral origin distance
            try:
                origin = neutral_origin.fit_neutral_origin(train_neutral)
                val_distance = neutral_origin.score_by_origin_distance(origin, val_matrix)
                threshold3a = neutral_origin.fit_threshold(val_distance, val_labels)
                val_pred3a = neutral_origin.predict_with_threshold(val_distance, threshold3a)
                candidates["neutral_origin_distance"].append(FittedMethodResult(
                    method_name="neutral_origin_distance", pooling_variant=pooling_variant, layer=layer,
                    predict_fn=lambda a, o=origin, t=threshold3a: neutral_origin.predict_with_threshold(
                        neutral_origin.score_by_origin_distance(o, a), t),
                    score_fn=lambda a, o=origin: neutral_origin.score_by_origin_distance(o, a),
                    val_accuracy=_accuracy(val_pred3a, val_labels),
                    reference_point=origin.origin, note="threshold fit on validation",
                ))
            except Exception as e:
                print(f"  neutral_origin_distance layer {layer} skipped: {e}")

            # Method 3b -- neutral origin direction
            try:
                direction3b = neutral_origin.fit_validation_direction(origin, val_harmful)
                val_scores3b = neutral_origin.score_by_validation_direction(origin, direction3b, val_matrix)
                threshold3b = neutral_origin.fit_threshold(val_scores3b, val_labels)
                val_pred3b = neutral_origin.predict_with_threshold(val_scores3b, threshold3b)
                candidates["neutral_origin_direction"].append(FittedMethodResult(
                    method_name="neutral_origin_direction", pooling_variant=pooling_variant, layer=layer,
                    predict_fn=lambda a, o=origin, d=direction3b, t=threshold3b: neutral_origin.predict_with_threshold(
                        neutral_origin.score_by_validation_direction(o, d, a), t),
                    score_fn=lambda a, o=origin, d=direction3b: neutral_origin.score_by_validation_direction(o, d, a),
                    val_accuracy=_accuracy(val_pred3b, val_labels),
                    reference_point=origin.origin,
                    note="direction AND threshold fit on validation split",
                ))
            except Exception as e:
                print(f"  neutral_origin_direction layer {layer} skipped: {e}")

            # Method 1 -- tone pole
            if include_tone_pole:
                try:
                    pole1 = tone_pole.fit_tone_pole(cache_dir, model_key, pooling_variant, layer)
                    val_pred1 = tone_pole.predict(pole1, val_matrix)
                    candidates["tone_pole"].append(FittedMethodResult(
                        method_name="tone_pole", pooling_variant=pooling_variant, layer=layer,
                        predict_fn=lambda a, p=pole1: tone_pole.predict(p, a),
                        score_fn=lambda a, p=pole1: tone_pole.score(p, a),
                        val_accuracy=_accuracy(val_pred1, val_labels),
                        reference_point=pole1.midpoint,
                    ))
                except Exception:
                    continue

    results = {}
    for method_name, method_candidates in candidates.items():
        if not method_candidates:
            continue
        best = max(method_candidates, key=lambda c: c.val_accuracy)
        test_matrix = activation_store.load_layer_matrix(
            cache_dir, model_key, formatting_variant, best.pooling_variant, best.layer, test_ids)
        test_pred   = best.predict_fn(test_matrix)
        test_scores = best.score_fn(test_matrix)
        ci = bootstrap_accuracy_ci(np.asarray(test_labels), test_pred, seed=seed)
        length_report = length_correlation_report(
            test_scores, [text_by_id[i] for i in test_ids],
            np.array([1.0 if lbl == "harmful" else 0.0 for lbl in test_labels]),
        )
        best.test_result = {**ci, "length_correlation": length_report}
        results[method_name] = best
    return results

_sta.run_subtest_a = _patched_run_subtest_a

# Now run
all_results = {}
all_b_results = {}
all_gap_summaries = {}
rows = []

for model_key, cache_dir in MODEL_CACHE_MAP.items():
    formatting_variants = [d.name for d in (cache_dir / model_key).iterdir()
                           if d.is_dir() and d.name != "generation"]
    for formatting_variant in formatting_variants:
        print(f"=== {model_key} / {formatting_variant} ===")
        a_results = _sta.run_subtest_a(cache_dir, model_key, formatting_variant)
        b_results = subtest_b_mod.run_subtest_b(cache_dir, model_key, formatting_variant, a_results)
        gap_summary = subtest_b_mod.summarize_a_b_gap(a_results, b_results)

        all_results[(model_key, formatting_variant)] = a_results
        all_b_results[(model_key, formatting_variant)] = b_results
        all_gap_summaries[(model_key, formatting_variant)] = gap_summary

        for method_name, fitted in a_results.items():
            gap = gap_summary.get(method_name, {})
            rows.append({
                "model": model_key, "formatting": formatting_variant, "method": method_name,
                "pooling_variant": fitted.pooling_variant, "layer": fitted.layer,
                "subtest_a_accuracy": fitted.test_result["accuracy"],
                "subtest_a_ci_low": fitted.test_result["ci_low"],
                "subtest_a_ci_high": fitted.test_result["ci_high"],
                "subtest_a_length_flagged": fitted.test_result["length_correlation"]["flagged"],
                "subtest_b_neutral_accuracy": gap.get("subtest_b_neutral_arm_accuracy"),
                "subtest_b_neutral_ci": gap.get("subtest_b_neutral_arm_ci"),
                "a_b_gap": gap.get("a_b_gap"),
                "note": fitted.note,
            })

results_df = pd.DataFrame(rows)
results_df

=== gpt2-medium / raw ===
=== qwen2.5-1.5b / chat ===
  neutral_origin_direction layer 0 skipped: fit_validation_direction: validation harmful mean equals the origin
=== qwen2.5-1.5b / raw ===
=== qwen2.5-1.5b-instruct / chat ===
  neutral_origin_direction layer 0 skipped: fit_validation_direction: validation harmful mean equals the origin
=== qwen2.5-1.5b-instruct / raw ===
=== llama-3.2-3b / raw ===
=== llama-3.2-3b-instruct / chat ===
  neutral_origin_direction layer 0 skipped: fit_validation_direction: validation harmful mean equals the origin
=== llama-3.2-3b-instruct / raw ===


,model,formatting,method,pooling_variant,layer,subtest_a_accuracy,subtest_a_ci_low,subtest_a_ci_high,subtest_a_length_flagged,subtest_b_neutral_accuracy,subtest_b_neutral_ci,a_b_gap,note
0,gpt2-medium,raw,content_pole,last_token,20,0.973684,0.934211,1.000000,False,0.687500,"(0.5416666666666666, 0.8125)",0.286184,
1,gpt2-medium,raw,neutral_origin_distance,last_token,12,0.684211,0.578947,0.789474,False,0.145833,"(0.0625, 0.25)",0.538377,threshold fit on validation
2,gpt2-medium,raw,neutral_origin_direction,last_token,21,0.947368,0.894737,0.986842,False,0.479167,"(0.3333333333333333, 0.625)",0.468202,direction AND threshold fit on validation split
3,qwen2.5-1.5b,chat,content_pole,masked_mean,27,0.907895,0.842105,0.973684,True,0.145833,"(0.041666666666666664, 0.25)",0.762061,
4,qwen2.5-1.5b,chat,neutral_origin_distance,masked_mean,18,0.842105,0.763158,0.921053,True,0.000000,"(0.0, 0.0)",0.842105,threshold fit on validation
5,qwen2.5-1.5b,chat,neutral_origin_direction,masked_mean,27,0.947368,0.894737,0.986842,True,0.208333,"(0.10416666666666667, 0.3333333333333333)",0.739035,direction AND threshold fit on validation split
6,qwen2.5-1.5b,chat,tone_pole,last_token,9,0.513158,0.394737,0.618750,False,0.020833,"(0.0, 0.0625)",0.492325,
7,qwen2.5-1.5b,raw,content_pole,last_token,27,0.960526,0.907895,1.000000,False,0.500000,"(0.3541666666666667, 0.6458333333333334)",0.460526,
8,qwen2.5-1.5b,raw,neutral_origin_distance,last_token,20,0.802632,0.697368,0.894737,False,0.000000,"(0.0, 0.0)",0.802632,threshold fit on validation
9,qwen2.5-1.5b,raw,neutral_origin_direction,last_token,0,0.592105,0.473684,0.697368,False,0.000000,"(0.0, 0.0)",0.592105,direction AND threshold fit on validation split


## Win condition: smallest A/B gap with competitive absolute accuracy on A

Per spec, this is decided per (model, formatting_variant) — do not presuppose Method 3 wins.

In [7]:
COMPETITIVE_A_THRESHOLD = 0.60  # edit if a different bar for "competitive" is wanted

winners = []
for (model_key, formatting_variant), gap_summary in all_gap_summaries.items():
    competitive = {m: g for m, g in gap_summary.items() if g["subtest_a_accuracy"] >= COMPETITIVE_A_THRESHOLD}
    pool = competitive or gap_summary
    if not pool:
        continue
    winner_name = min(pool, key=lambda m: abs(pool[m]["a_b_gap"]))
    winners.append({
        "model": model_key, "formatting": formatting_variant, "winning_method": winner_name,
        **pool[winner_name],
    })

winners_df = pd.DataFrame(winners)
winners_df

,model,formatting,winning_method,subtest_a_accuracy,subtest_a_ci,subtest_b_neutral_arm_accuracy,subtest_b_neutral_arm_ci,a_b_gap
0,gpt2-medium,raw,content_pole,0.973684,"(0.9342105263157895, 1.0)",0.687500,"(0.5416666666666666, 0.8125)",0.286184
1,qwen2.5-1.5b,chat,neutral_origin_direction,0.947368,"(0.8947368421052632, 0.9868421052631579)",0.208333,"(0.10416666666666667, 0.3333333333333333)",0.739035
2,qwen2.5-1.5b,raw,content_pole,0.960526,"(0.9078947368421053, 1.0)",0.500000,"(0.3541666666666667, 0.6458333333333334)",0.460526
3,qwen2.5-1.5b-instruct,chat,content_pole,0.973684,"(0.9342105263157895, 1.0)",0.958333,"(0.8958333333333334, 1.0)",0.015351
4,qwen2.5-1.5b-instruct,raw,tone_pole,0.855263,"(0.7763157894736842, 0.9342105263157895)",0.500000,"(0.3541666666666667, 0.6458333333333334)",0.355263
5,llama-3.2-3b,raw,content_pole,0.960526,"(0.9078947368421053, 1.0)",0.437500,"(0.3125, 0.5833333333333334)",0.523026
6,llama-3.2-3b-instruct,chat,content_pole,0.986842,"(0.9605263157894737, 1.0)",0.770833,"(0.6458333333333334, 0.875)",0.216009
7,llama-3.2-3b-instruct,raw,tone_pole,0.815789,"(0.7236842105263158, 0.8947368421052632)",0.520833,"(0.3958333333333333, 0.6666666666666666)",0.294956


## Probe vs. cosine/mean-diff (the second open method question)

Run for whichever method won each (model, formatting_variant) above, at the layer/pooling variant that method already selected on validation. Reported as a comparison, not a replacement.

In [8]:
from src.methods import probe as probe_mod

probe_rows = []
for winner in winners:
    key = (winner["model"], winner["formatting"])
    fitted = all_results[key][winner["winning_method"]]

    splits_df_local = splits.load_frozen_splits()
    train_ids_l, train_labels_l = _ids_labels("train")
    test_ids_l, test_labels_l = _ids_labels("test")

    cache_dir = MODEL_CACHE_MAP[winner["model"]]
    train_matrix = activation_store.load_layer_matrix(
        cache_dir, winner["model"], winner["formatting"], fitted.pooling_variant, fitted.layer, train_ids_l
    )
    test_matrix = activation_store.load_layer_matrix(
        cache_dir, winner["model"], winner["formatting"], fitted.pooling_variant, fitted.layer, test_ids_l
    )
    probe_result = probe_mod.fit_and_score_probe(
        train_matrix, np.array(train_labels_l), test_matrix, np.array(test_labels_l)
    )
    probe_rows.append({
        "model": winner["model"], "formatting": winner["formatting"], "method": winner["winning_method"],
        "cosine_meandiff_accuracy": winner["subtest_a_accuracy"],
        "probe_accuracy": probe_result["accuracy"],
    })

pd.DataFrame(probe_rows)

,model,formatting,method,cosine_meandiff_accuracy,probe_accuracy
0,gpt2-medium,raw,content_pole,0.973684,1.000000
1,qwen2.5-1.5b,chat,neutral_origin_direction,0.947368,0.986842
2,qwen2.5-1.5b,raw,content_pole,0.960526,0.986842
3,qwen2.5-1.5b-instruct,chat,content_pole,0.973684,1.000000
4,qwen2.5-1.5b-instruct,raw,tone_pole,0.855263,1.000000
5,llama-3.2-3b,raw,content_pole,0.960526,1.000000
6,llama-3.2-3b-instruct,chat,content_pole,0.986842,1.000000
7,llama-3.2-3b-instruct,raw,tone_pole,0.815789,0.986842


## Random-direction null check (spec requirement 4)

In [9]:
control_rows = []
for (model_key, formatting_variant), a_results in all_results.items():
    for method_name, fitted in a_results.items():
        control = subtest_a_mod.evaluate_random_direction_control(MODEL_CACHE_MAP[model_key], model_key, formatting_variant, fitted)
        control_rows.append({
            "model": model_key, "formatting": formatting_variant, "method": method_name,
            "fitted_accuracy": control["fitted_method_test_accuracy"],
            "random_direction_mean_accuracy": control["mean_accuracy"],
            "random_direction_range": (control["min_accuracy"], control["max_accuracy"]),
        })

pd.DataFrame(control_rows)

,model,formatting,method,fitted_accuracy,random_direction_mean_accuracy,random_direction_range
0,gpt2-medium,raw,content_pole,0.973684,0.509868,"(0.32894736842105265, 0.6710526315789473)"
1,gpt2-medium,raw,neutral_origin_distance,0.684211,0.481579,"(0.3026315789473684, 0.5789473684210527)"
2,gpt2-medium,raw,neutral_origin_direction,0.947368,0.499342,"(0.2894736842105263, 0.6447368421052632)"
3,qwen2.5-1.5b,chat,content_pole,0.907895,0.506579,"(0.19736842105263158, 0.7105263157894737)"
4,qwen2.5-1.5b,chat,neutral_origin_distance,0.842105,0.449342,"(0.23684210526315788, 0.7763157894736842)"
5,qwen2.5-1.5b,chat,neutral_origin_direction,0.947368,0.508553,"(0.2631578947368421, 0.7105263157894737)"
6,qwen2.5-1.5b,chat,tone_pole,0.513158,0.486842,"(0.39473684210526316, 0.5394736842105263)"
7,qwen2.5-1.5b,raw,content_pole,0.960526,0.490132,"(0.3026315789473684, 0.7763157894736842)"
8,qwen2.5-1.5b,raw,neutral_origin_distance,0.802632,0.488816,"(0.34210526315789475, 0.631578947368421)"
9,qwen2.5-1.5b,raw,neutral_origin_direction,0.592105,0.511842,"(0.39473684210526316, 0.6447368421052632)"


## Sub-test B harmful arm — qualitative only, N=3, never a headline number

Per data/subtest_b_MANIFEST.md and explicit instruction: printed separately, never merged into the table above, never bootstrap-CI'd.

In [10]:
for (model_key, formatting_variant), b_results in all_b_results.items():
    for method_name, arms in b_results.items():
        qual = arms["harmful_arm_qualitative"]
        n_correct = sum(item["correct"] for item in qual["per_item"])
        print(f"{model_key} / {formatting_variant} / {method_name}: {n_correct}/{len(qual['per_item'])} correct on "
              f"the N={qual['n_pairs']} verified-clean harmful arm -- {qual['note']}")

gpt2-medium / raw / content_pole: 5/6 correct on the N=3 verified-clean harmful arm -- N=3 verified-clean sourced pairs (6 items). Bootstrap CI at this N is uninformative (binomial 95% CI for 3/3 correct spans roughly [29%, 100%]) -- see data/subtest_b_MANIFEST.md. Qualitative case-study supplement only; do not cite an accuracy percentage from this block as a result.
gpt2-medium / raw / neutral_origin_distance: 6/6 correct on the N=3 verified-clean harmful arm -- N=3 verified-clean sourced pairs (6 items). Bootstrap CI at this N is uninformative (binomial 95% CI for 3/3 correct spans roughly [29%, 100%]) -- see data/subtest_b_MANIFEST.md. Qualitative case-study supplement only; do not cite an accuracy percentage from this block as a result.
gpt2-medium / raw / neutral_origin_direction: 6/6 correct on the N=3 verified-clean harmful arm -- N=3 verified-clean sourced pairs (6 items). Bootstrap CI at this N is uninformative (binomial 95% CI for 3/3 correct spans roughly [29%, 100%]) -- see

In [11]:
output_path = pathlib.Path("/kaggle/working/method_comparison_results.csv")
results_df.to_csv(output_path, index=False)
print("wrote", output_path.resolve())

wrote /kaggle/working/method_comparison_results.csv
